# Deriva de datos y degradación compatible con deriva conceptual
Los datos y las métricas se leen desde Snowflake. El replay parte de ocho lotes de 250 lecturas de la partición congelada. El lote 6 añade 18 Nm al torque.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
from monitoreo.snowflake_store import read_rows
source = read_rows("monitoreo", "alertas")
lots = pd.DataFrame(source["monitoreo"]).sort_values("periodo")
assert list(lots.periodo) == list(range(1,9))
display(lots[["periodo","n","psi_max","ap","caida_ap","alert_rate","cost_per_1000","costo_regla"]])
display(pd.DataFrame(source["alertas"]))


/usr/local/lib/python3.12/site-packages/snowflake/connector/options.py:108: UserWarning: You have an incompatible version of 'pyarrow' installed (19.0.1), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(
/usr/local/lib/python3.12/site-packages/snowflake/connector/vendored/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.18) or chardet (7.7.0)/charset_normalizer (3.5.3) doesn't match a supported version!
  warnings.warn(


,periodo,n,psi_max,ap,caida_ap,alert_rate,cost_per_1000,costo_regla
0,1,250,0.056317,0.940000,False,0.052,112000,194000
1,2,250,0.072041,0.961735,False,0.036,60000,164000
2,3,250,0.049605,1.000000,False,0.044,64000,234000
3,4,250,0.051837,1.000000,False,0.052,68000,174000
4,5,250,0.063971,0.965152,False,0.072,96000,206000
5,6,250,2.632994,0.668493,True,0.624,404000,340000
6,7,250,0.066788,0.875962,False,0.040,94000,190000
7,8,250,0.044110,0.706717,True,0.036,144000,198000


,origen,periodo,psi,query_id,responsable,severidad,variable
0,replay_pset4,6,2.632994,01c8a18f-0001-dff7-0001-77be0026008a,Datos y Mantenimiento,alta,torque_nm


In [2]:
controls = lots[lots.periodo != 6]
induced = lots.loc[lots.periodo == 6,"psi_max"].item()
cut = .25
assert controls.psi_max.max() < cut < induced
print(f"PSI máximo de control: {controls.psi_max.max():.4f}; PSI inducido: {induced:.4f}; corte: {cut}")
display(lots.loc[lots.caida_ap & ~lots.deriva, ["periodo","psi_max","ap"]])


PSI máximo de control: 0.0720; PSI inducido: 2.6330; corte: 0.25


,periodo,psi_max,ap
7,8,0.04411,0.706717


## Interpretación
El torque alterado en el lote 6 demuestra deriva de datos al cambiar la distribución de una entrada. Una caída de AP con PSI bajo es una señal compatible con deriva conceptual. También puede surgir por el tamaño del lote, la composición de fallas o errores de etiqueta.

In [3]:
import numpy as np
from sklearn.metrics import average_precision_score
from monitoreo.snowflake_store import read_partition
spec = json.loads(Path("docs/split.json").read_text(encoding="utf-8"))
_, held, provenance = read_partition(spec)
pred = pd.DataFrame(read_rows("predicciones")["predicciones"])
period = pred[(pred.origen=="replay_pset4") & (pred.periodo==1)]
joined = period.merge(held[["reading_id","machine_failure"]], on="reading_id", validate="one_to_one")
score = joined.probabilidad.to_numpy()
labels = joined.machine_failure.to_numpy()
# Escenario deliberado: asignar fallas a los puntajes más bajos, sin cambiar sensores ni prevalencia.
changed = np.zeros_like(labels)
changed[np.argsort(score)[:labels.sum()]]=1
print({"query_id":provenance["query_id"], "ap_original":average_precision_score(labels,score),
       "ap_relacion_alterada":average_precision_score(changed,score),"cambio_psi":0.0})


{'query_id': '01c8a190-0001-dfb3-0001-77be00259242', 'ap_original': np.float64(0.9400000000000001), 'ap_relacion_alterada': np.float64(0.022269409300736197), 'cambio_psi': 0.0}
